# 07 — De texto a respuesta: qué pasa dentro de un modelo de lenguaje

**Texto → Tokens → Vectores → Transformer → Logits → Token → Texto**

Hasta aquí el diplomado trabajó los dos extremos de esa cadena y dejó el centro como una caja
negra. En la Sesión 10 vimos cómo un texto se convierte en vectores (*embeddings*) y cómo se
comparan entre sí; en las sesiones que siguen vamos a mandarle texto a un modelo grande por una
API y a recibir texto de vuelta. Este notebook abre la caja de en medio: **un modelo de lenguaje
completo, corriendo en tu computadora, del que podemos ver todas las piezas.**

No hay API ni llave aquí: descargamos un modelo pequeño y lo desarmamos.

## ¿Qué vas a poder hacer al terminar?

1. Explicar por qué el modelo nunca ve letras ni palabras, sino **tokens**, y por qué eso se
   traduce en dinero y en límites de longitud.
2. Localizar los *embeddings* de la Sesión 10 **dentro** del modelo, y distinguir el vector de
   entrada (fijo) del vector que sale de las capas (contextual).
3. Leer la salida real del modelo: una **distribución de probabilidad sobre las 50,257 piezas del
   vocabulario**, no una frase.
4. Explicar qué hace exactamente el parámetro `temperature` que vas a usar en la API, viendo cómo
   deforma esa distribución.
5. Escribir a mano el **bucle autorregresivo** y comprobar que reproduce a `model.generate()`.
6. Saber qué cambia y qué no cuando el modelo es 500 veces más grande y vive en el servidor de
   otra empresa.

## Prerrequisitos

Sesión 9 (redes neuronales: capas, pesos, *softmax*) y Sesión 10 (*embeddings*, similitud coseno).
De las notas del módulo, la sección *"¿Cómo funciona un LLM? Breve introducción a la arquitectura
Transformer"* del capítulo 2 es el respaldo teórico de este notebook: aquí tocamos con las manos
lo que ahí está escrito en fórmulas.

> **Descarga.** El modelo pesa unos **500 MB** y se baja una sola vez. Si estás en el salón,
> corre la sección 0 apenas empiece la sesión.

## 0. Preparación del entorno

La primera celda instala solo lo que falte (necesaria en Google Colab). La segunda importa y
descarga el modelo. **Corre las dos antes que cualquier otra cosa.**

> **Nota técnica importante.** La línea `os.environ["USE_TF"] = "0"` debe ejecutarse *antes* de
> importar `transformers`. Esa librería busca TensorFlow al importarse, y en varias computadoras
> (procesadores sin instrucciones AVX, instalaciones incompletas de TensorFlow) esa búsqueda tumba
> el *kernel* de Jupyter sin dar un mensaje de error claro. Como aquí trabajamos con PyTorch, le
> decimos explícitamente que no lo busque. Es la misma nota de los notebooks de la Sesión 10.

In [ ]:
# --- Dependencias -------------------------------------------------------------
# Instala SOLO lo que falte, así que puedes correrla siempre.
# En Google Colab `transformers`, `torch` y `pandas` ya vienen instalados.
import importlib.util, subprocess, sys

EN_COLAB = "google.colab" in sys.modules

def asegurar(paquete, modulo=None):
    """Instala `paquete` solo si su módulo no está disponible.

    Usamos find_spec y no un `import` dentro de un try: find_spec NO ejecuta el
    módulo, y ejecutar `transformers` antes de fijar USE_TF puede tumbar el kernel
    (ver la nota de la celda anterior).
    """
    modulo = modulo or paquete.replace("-", "_")
    if importlib.util.find_spec(modulo) is None:
        print(f"Instalando {paquete} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", paquete])

for paquete, modulo in [("transformers", "transformers"),
                        ("torch",        "torch"),
                        ("pandas",       "pandas")]:
    asegurar(paquete, modulo)

print("Dependencias listas." + ("   (Google Colab detectado)" if EN_COLAB else ""))

In [ ]:
import os
os.environ["USE_TF"] = "0"                  # SIEMPRE antes de importar transformers
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import warnings; warnings.filterwarnings("ignore")
import pandas as pd
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM

torch.manual_seed(42)

# GPT-2 pequeño entrenado en español. ~500 MB, se descarga una sola vez.
MODELO = "datificate/gpt2-small-spanish"

try:
    tok = AutoTokenizer.from_pretrained(MODELO)
    modelo = AutoModelForCausalLM.from_pretrained(MODELO)
except Exception as e:                       # plan B si ese repositorio no responde
    print(f"No se pudo cargar {MODELO} ({type(e).__name__}). Uso 'gpt2' (inglés).")
    print("El notebook corre igual, pero los NÚMEROS del texto no van a coincidir.")
    MODELO = "gpt2"
    tok = AutoTokenizer.from_pretrained(MODELO)
    modelo = AutoModelForCausalLM.from_pretrained(MODELO)

modelo.eval()                                # modo evaluación: sin dropout, sin entrenar
print("Modelo cargado:", MODELO)

In [ ]:
cfg = modelo.config
n_par = sum(p.numel() for p in modelo.parameters())

print(f"Parámetros ...................... {n_par/1e6:,.1f} millones")
print(f"Capas del Transformer ........... {cfg.n_layer}")
print(f"Dimensión de cada vector (d) .... {cfg.n_embd}")
print(f"Cabezas de atención por capa .... {cfg.n_head}")
print(f"Tamaño del vocabulario .......... {cfg.vocab_size:,} tokens")
print(f"Ventana de contexto ............. {cfg.n_positions:,} tokens")

Es un modelo **chico**: 124 millones de parámetros. `openai/gpt-oss-120b`, el que usamos por la
API en los notebooks siguientes, tiene 117 **mil** millones — unas 940 veces más. La arquitectura,
sin embargo, es la misma en lo esencial, y esa es la razón de usar el chico: cabe en tu
computadora y podemos verle todas las piezas.

Fíjate en la última línea: **la ventana de contexto se mide en tokens**, no en palabras ni en
páginas. En la sección 1 vemos qué es exactamente un token, y en la 7 por qué ese número es el
límite duro de cualquier aplicación que construyas.

## 1. Texto → tokens

El modelo no procesa letras ni palabras: procesa **tokens**, piezas de texto de tamaño variable
que se fijaron al entrenar. Cada token tiene un número de identidad (su *id*) dentro de un
vocabulario cerrado. Lo primero que pasa con tu texto es que se parte en esas piezas.

In [ ]:
frase = "La regresión logística ordinal es un modelo"

piezas = tok.tokenize(frase)
ids    = tok(frase)["input_ids"]

tabla = pd.DataFrame({
    "token (como lo guarda el modelo)": piezas,
    "id": ids,
    "texto que representa": [repr(tok.decode([i])) for i in ids],
})
print(f"{len(frase.split())} palabras  ->  {len(ids)} tokens\n")
tabla

Tres cosas que vale la pena mirar en esa tabla:

- **El símbolo `Ġ` es un espacio.** El token `Ġmodelo` no es lo mismo que `modelo`: el primero
  incluye el espacio que lo precede. Por eso el modelo no necesita una regla aparte para saber
  dónde empiezan las palabras.
- **Las palabras raras se parten en pedazos.** `ordinal` no existe como token: se arma con
  ` or` + `d` + `inal`. El vocabulario es cerrado (50,257 piezas), así que cualquier palabra que
  no esté completa se reconstruye con trozos. Esto es exactamente lo que viste con los subtokens
  `##` de BERT en el notebook 05 de la Sesión 10; aquí la notación cambia, la idea no.
- **Nada se pierde.** `tok.decode` devuelve el texto original. La tokenización es reversible.

### 1.1 Por qué esto te va a costar dinero

Los tokens se aprenden del corpus de entrenamiento: lo que aparece seguido en ese corpus se
vuelve una pieza propia, y lo que casi no aparece se arma por trozos. Un tokenizador entrenado
en español y uno entrenado en inglés parten el **mismo texto** de forma muy distinta.

In [ ]:
from transformers import AutoTokenizer
tok_en = AutoTokenizer.from_pretrained("gpt2")   # tokenizador entrenado en inglés (solo 1 MB)

pares = [
    ("El agrupamiento por k-medias necesita elegir el número de grupos.",
     "Clustering with k-means requires choosing the number of groups."),
    ("La estimación por máxima verosimilitud requiere derivar la log-verosimilitud.",
     "Maximum likelihood estimation requires differentiating the log-likelihood."),
]

filas = []
for es, en in pares:
    filas.append({
        "oración (español)": es[:42] + "...",
        "tokens con el tokenizador ES": len(tok(es)["input_ids"]),
        "tokens con el tokenizador EN": len(tok_en(es)["input_ids"]),
        "la misma idea en inglés, tokenizador EN": len(tok_en(en)["input_ids"]),
    })
display(pd.DataFrame(filas))

print("\n'verosimilitud' partida por cada tokenizador:")
print("  entrenado en español:", tok.tokenize(" verosimilitud"))
print("  entrenado en inglés :", tok_en.tokenize(" verosimilitud"))

El mismo texto en español cuesta **15 tokens** con un tokenizador entrenado en español y **25**
con uno entrenado en inglés: un 67 % más caro. Y la misma idea escrita en inglés cuesta 15 tokens
con el tokenizador inglés. La palabra `verosimilitud` se arma con 3 piezas en uno y con 5 en el
otro.

Esto no es una curiosidad de laboratorio. Los modelos comerciales grandes (GPT, Llama, Claude)
se entrenaron con corpus mayoritariamente en inglés, así que **tu texto en español consume más
tokens que su traducción al inglés**. Las consecuencias son concretas:

| Consecuencia | Por qué |
|---|---|
| Cuesta más | Los proveedores cobran por token de entrada y de salida. |
| Cabe menos | La ventana de contexto se mide en tokens: el mismo documento en español llena antes el límite. |
| Se corta antes | `max_tokens` limita la respuesta en tokens, no en palabras. |

Es también la explicación mecánica de algo que ya mediste en el notebook 04 de la Sesión 10: un
modelo que parte el español en pedacitos sin sentido difícilmente va a representarlo bien.

## 2. Tokens → vectores

Un *id* es un número de catálogo: el 2702 no es "más grande" que el 552 en ningún sentido útil.
Para que el modelo pueda operar, cada id se cambia por un vector. **Esa tabla de vectores es el
*embedding* de la Sesión 10, y vive literalmente dentro del modelo**: es la matriz `wte`
(*word token embeddings*).

In [ ]:
wte = modelo.transformer.wte.weight     # una fila por token del vocabulario
wpe = modelo.transformer.wpe.weight     # una fila por posición posible

print("wte (tabla de tokens)    :", tuple(wte.shape),
      f"->  {wte.numel()/1e6:,.1f}M parámetros, {100*wte.numel()/n_par:.0f}% del modelo entero")
print("wpe (tabla de posiciones):", tuple(wpe.shape))

id_regresion = tok(" regresión")["input_ids"][0]
v = wte[id_regresion]
print(f"\nToken {tok.decode([id_regresion])!r} (id {id_regresion})")
print("  vector de", v.shape[0], "dimensiones, norma %.3f" % v.norm())
print("  primeras 8 coordenadas:", [round(float(x), 3) for x in v[:8]])

Es una **tabla de búsqueda**, no un cálculo: convertir un token en vector es ir al renglón que le
toca. Y no es un detalle menor del modelo — esa sola tabla es el 31 % de todos sus parámetros.

Dos avisos para no confundir esto con lo de la Sesión 10:

1. Este vector es **fijo**: el renglón de ` regresión` es el mismo aparezca donde aparezca. Es un
   embedding *estático*, como los del notebook 04.
2. Por lo tanto **todavía no es el embedding contextual** de BERT ni el de una oración. Es apenas
   el punto de partida. Lo que lo vuelve contextual son las 12 capas de la sección 3.

Antes de eso falta una pieza: con solo `wte`, la frase *"el perro mordió al cartero"* y
*"el cartero mordió al perro"* entrarían al modelo como el mismo conjunto de vectores. Por eso se
suma un segundo vector que depende de **la posición**.

In [ ]:
entrada_pos0 = wte[id_regresion] + wpe[0]    # el token en la posición 0
entrada_pos5 = wte[id_regresion] + wpe[5]    # EL MISMO token en la posición 5

print("coseno entre el mismo token en dos posiciones distintas: %.4f"
      % F.cosine_similarity(entrada_pos0, entrada_pos5, dim=0))
print("(sin el vector de posición sería exactamente 1.0000)")

0.43: mover una palabra de lugar cambia su vector de entrada casi tanto como cambiarla de palabra.
El orden entra al modelo por aquí, sumado al vector del token. Eso es la *codificación posicional*
de las notas.

## 3. El Transformer: 12 capas que reescriben los vectores

Ya tenemos una matriz de vectores de entrada (uno por token). El bloque Transformer la recibe y la
devuelve **con las mismas dimensiones, pero con los vectores reescritos**: cada token mira a los
tokens anteriores —eso es la *atención*— y actualiza su vector con lo que encuentra. Eso ocurre 12
veces seguidas.

Con `output_hidden_states=True` podemos ver el resultado de cada capa.

In [ ]:
texto = "Pedí un préstamo en el banco porque necesitaba dinero"
enc = tok(texto, return_tensors="pt")

with torch.no_grad():
    salida = modelo(**enc, output_hidden_states=True)

print("tokens de entrada:", enc["input_ids"].shape[1])
print("estados guardados:", len(salida.hidden_states), "(la entrada + una por cada capa)")
print("forma de cada estado:", tuple(salida.hidden_states[0].shape), "= (lote, tokens, dimensión)")

### 3.1 El mismo token, dos significados

`banco` es la palabra de siempre para esta prueba: el del dinero y el del parque. Vamos a ponerla
**en la misma posición** en tres oraciones —dos con el sentido financiero y una con el del
parque— y a seguir su vector capa por capa.

Que esté en la misma posición importa: así el vector de entrada es *idéntico* en los tres casos
(mismo token, misma posición) y cualquier diferencia posterior viene del contexto, no del lugar.

In [ ]:
oraciones = {
    "financiero" : "Pedí un préstamo en el banco porque necesitaba dinero",
    "parque"     : "Me senté a descansar en el banco porque estaba cansado",
    "financiero2": "Solicité el crédito en el banco porque no tenía efectivo",
}

id_banco = tok(" banco")["input_ids"][0]

def vectores_por_capa(texto):
    """Devuelve el vector del token ' banco' en cada capa, y su posición."""
    enc = tok(texto, return_tensors="pt")
    posicion = enc["input_ids"][0].tolist().index(id_banco)
    with torch.no_grad():
        hs = modelo(**enc, output_hidden_states=True).hidden_states
    return [h[0, posicion] for h in hs], posicion

vecs, pos = {}, {}
for nombre, texto in oraciones.items():
    vecs[nombre], pos[nombre] = vectores_por_capa(texto)
print("posición de ' banco' en cada oración:", pos)

filas = []
for capa in range(len(vecs["financiero"])):
    filas.append({
        "capa": "entrada" if capa == 0 else capa,
        "financiero vs. parque"      : round(float(F.cosine_similarity(
            vecs["financiero"][capa], vecs["parque"][capa], dim=0)), 3),
        "financiero vs. financiero2" : round(float(F.cosine_similarity(
            vecs["financiero"][capa], vecs["financiero2"][capa], dim=0)), 3),
    })
pd.DataFrame(filas)

Léelo de arriba hacia abajo, que es como corre el modelo:

- **En la entrada los tres vectores son idénticos** (coseno 1.000, sin redondeo). Para el modelo,
  en ese momento, `banco` es `banco`: una entrada de catálogo sin significado propio.
- Las capas de en medio los separan muy poco (0.99).
- **En la última capa los dos sentidos ya están separados**: 0.707 entre el banco del dinero y el
  del parque, contra 0.946 entre los dos bancos financieros. El vector del token dejó de
  representar la palabra y pasó a representar *esta palabra en esta oración*.

Eso es un **embedding contextual**, y es lo mismo que mediste con BERT en el notebook 05. Ahí la
separación fue más marcada (0.831 contra 0.46) porque BERT está entrenado exactamente para eso y
puede mirar hacia ambos lados de la oración; aquí el modelo es chico y solo mira hacia atrás, así
que la separación es más modesta. La dirección del resultado, que es lo que importa, es la misma.

### 3.2 Solo se mira hacia atrás (atención causal)

Un modelo generativo tiene una restricción que BERT no tiene: al calcular el vector del token $i$
solo puede usar los tokens $1 \dots i$, nunca los que vienen después. Si pudiera ver el futuro,
predecir la siguiente palabra sería copiarla.

No hay que creerlo: se comprueba. Alarguemos una frase por la derecha y veamos si cambian los
vectores de los tokens que ya estaban.

In [ ]:
corta = "El modelo de regresión"
larga = "El modelo de regresión lineal explica muy bien los datos"

with torch.no_grad():
    h_corta = modelo(**tok(corta, return_tensors="pt"), output_hidden_states=True).hidden_states[-1]
    h_larga = modelo(**tok(larga, return_tensors="pt"), output_hidden_states=True).hidden_states[-1]

n = h_corta.shape[1]
dif = (h_corta[0, :n] - h_larga[0, :n]).abs().max()
print(f"Los primeros {n} tokens son los mismos en ambas frases.")
print(f"Diferencia máxima entre sus vectores finales: {dif:.2e}")
print("(cero hasta el error de redondeo de los flotantes)")

Idénticos. Agregar texto al final no cambia nada de lo que ya estaba calculado.

De aquí salen dos cosas que vas a usar después:

- Es la razón técnica de que un LLM "prediga la siguiente palabra": la arquitectura no le permite
  hacer otra cosa.
- Es también la razón de que los proveedores puedan cachear el inicio del *prompt*. Como el
  principio no cambia cuando agregas texto al final, el trabajo hecho sobre él se reutiliza. Por
  eso conviene poner lo estable —tu prompt de sistema— al principio, y lo que varía al final.

## 4. Vectores → logits: la salida real del modelo

Al final de las 12 capas tenemos un vector de 768 dimensiones por token. La última pieza del
modelo, `lm_head`, proyecta ese vector a **un número por cada token del vocabulario**: 50,257
números llamados *logits*.

El que importa para generar es el del **último** token: es el único que vio toda la frase.

In [ ]:
print("¿lm_head reusa la tabla de embeddings wte?", torch.equal(modelo.lm_head.weight, wte))

Sí: es la misma matriz. A la entrada se usa como tabla de búsqueda (id → vector) y a la salida
como medida de parecido (vector → un logit por token). Se le llama *weight tying* y ahorra los
38.6 millones de parámetros de tener dos tablas.

In [ ]:
prompt = "La regresión logística ordinal es un modelo que"
enc = tok(prompt, return_tensors="pt")

with torch.no_grad():
    logits_todos = modelo(**enc).logits

print("forma de los logits:", tuple(logits_todos.shape), "= (lote, tokens, vocabulario)")

logits = logits_todos[0, -1]                 # solo el último token
probs  = torch.softmax(logits, dim=-1)       # logits -> probabilidades

mejores = torch.topk(probs, 10)
pd.DataFrame({
    "siguiente token": [repr(tok.decode([i])) for i in mejores.indices],
    "probabilidad": [round(float(p), 4) for p in mejores.values],
    "": ["█" * int(round(float(p) * 120)) for p in mejores.values],
})

**Ésta es la salida del modelo.** No una frase, no una respuesta: una distribución de probabilidad
sobre las 50,257 piezas del vocabulario, obtenida con la misma *softmax* que usaste en la Sesión 9
para clasificar dígitos. La diferencia es el número de clases: 10 allá, 50,257 aquí.

Y está lejos de ser una decisión obvia:

In [ ]:
print("probabilidad del token más probable : %.3f" % mejores.values[0])
print("masa acumulada en los 10 mejores    : %.3f" % mejores.values.sum())
print("masa repartida entre los otros %d : %.3f" % (50257 - 10, 1 - mejores.values.sum()))

entropia = -(probs * torch.log(probs + 1e-12)).sum()
orden = torch.sort(probs, descending=True).values
print("\nentropía de la distribución          : %.2f nats" % entropia)
print("tokens necesarios para juntar el 90%% : %d" % (int((torch.cumsum(orden, 0) < 0.9).sum()) + 1))

El 58 % de la probabilidad está repartido fuera del *top* 10, y hacen falta 396 tokens distintos
para juntar el 90 % de la masa. **El modelo no sabe qué sigue; tiene una opinión difusa.** Todo lo
que viene después —la temperatura, el muestreo, y buena parte de las alucinaciones— sale de esta
observación.

## 5. Elegir un token: temperatura y muestreo

Alguien tiene que convertir esa distribución en **un** token. Ese alguien no es el modelo: es el
código que lo envuelve, y es donde entran los parámetros que ya viste en la API.

La **temperatura** $T$ divide los logits antes de la *softmax*:

$$P(\text{token}_j) = \frac{e^{z_j / T}}{\sum_i e^{z_i / T}}$$

Con $T$ chica las diferencias se agrandan y la distribución se concentra; con $T$ grande se
aplana. Fíjate que la temperatura **no aporta información**: solo deforma lo que el modelo ya
había calculado.

In [ ]:
filas = []
for T in [0.2, 0.7, 1.0, 1.5]:
    p = torch.softmax(logits / T, dim=-1)
    cinco = torch.topk(p, 5)
    orden = torch.sort(p, descending=True).values
    filas.append({
        "T": T,
        **{tok.decode([i]).strip(): round(float(v), 3)
           for v, i in zip(cinco.values, cinco.indices)},
        "entropía": round(float(-(p * torch.log(p + 1e-12)).sum()), 2),
        "tokens para el 90%": int((torch.cumsum(orden, 0) < 0.9).sum()) + 1,
    })
pd.DataFrame(filas).set_index("T")

La última columna es la que más dice. Con $T = 0.2$ un solo token se lleva el 90 % de la
probabilidad: el modelo se vuelve casi determinista. Con $T = 1.5$ hacen falta **18,979** tokens
para juntar esa misma masa, y entre ellos hay basura. Por eso `temperature=1.5` no produce
"creatividad": produce texto incoherente.

Dos estrategias más, que conviene no confundir:

- **Greedy** (`do_sample=False`): siempre el token más probable. Reproducible, pero se atora en
  repeticiones.
- **Top-$p$ / *nucleus*** (`top_p=0.9`): recorta la cola quedándose con los tokens que acumulan el
  90 % de la probabilidad, y muestrea solo entre ellos. Es el cinturón de seguridad que evita que
  una temperatura alta saque un token absurdo.

En la app de la sesión 11 fijamos `temperature=0.2` precisamente por esto: queremos que el modelo
se pegue al contexto recuperado, no que explore.

In [ ]:
def generar(prompt, n=18, **kw):
    enc = tok(prompt, return_tensors="pt")
    with torch.no_grad():
        salida = modelo.generate(**enc, max_new_tokens=n,
                                 pad_token_id=tok.eos_token_id, **kw)
    return tok.decode(salida[0], skip_special_tokens=True)

for T in [0.3, 1.0]:
    print(f"--- temperature = {T} " + "-" * 40)
    torch.manual_seed(11)
    for _ in range(3):
        print("  ", generar(prompt, do_sample=True, temperature=T, top_p=0.95))

Tres corridas con **el mismo prompt y el mismo modelo** dan tres textos distintos. Esa es la
*variabilidad* que las notas mencionan como riesgo de construir sistemas sobre un LLM, y aquí se
ve de dónde sale: no es un defecto del modelo, es que alguien está tirando un dado cargado con su
distribución.

## 6. El bucle autorregresivo

Ya tenemos un token. ¿Cómo se llega a un párrafo? Se pega al final del texto y se vuelve a
empezar. Nada más. Escribámoslo a mano.

In [ ]:
secuencia = tok(prompt, return_tensors="pt")["input_ids"]

print(prompt)
for paso in range(12):
    with torch.no_grad():
        lg = modelo(secuencia).logits[0, -1]     # 1) predecir con TODO lo que hay
    siguiente = int(lg.argmax())                 # 2) elegir (aquí: greedy)
    p = float(torch.softmax(lg, -1)[siguiente])
    print(f"  paso {paso+1:2d}: {tok.decode([siguiente])!r:16s} (p = {p:.3f})")
    # 3) pegarlo al final y repetir
    secuencia = torch.cat([secuencia, torch.tensor([[siguiente]])], dim=1)

texto_manual = tok.decode(secuencia[0])
print("\nresultado:", texto_manual)

In [ ]:
texto_generate = generar(prompt, n=12, do_sample=False)

print("¿nuestro bucle == model.generate()?", texto_manual == texto_generate)
print(texto_generate)

Idénticos. `model.generate()` es ese bucle de tres líneas con más opciones de muestreo y varias
optimizaciones encima; no hay nada más.

Tres consecuencias que explican cosas que ya viste o vas a ver:

1. **La respuesta se escribe de izquierda a derecha, sin plan.** El modelo no sabe en la palabra 3
   cómo va a terminar la oración. Es la raíz de que un LLM se comprometa con una afirmación falsa
   y luego la sostenga: ya la escribió.
2. **Generar es caro; leer el prompt es barato.** El prompt entero se procesa de una sola pasada,
   pero cada token nuevo cuesta una pasada completa por el modelo. Por eso `max_tokens` es el
   parámetro que más mueve la latencia, y por eso los proveedores cobran más caro el token de
   salida que el de entrada.
3. **Se puede cachear.** Como vimos en 3.2, lo ya calculado no cambia al agregar texto al final;
   guardar esos cálculos (*KV cache*) es lo que hace viable generar cientos de tokens.

## 7. Lo mismo, 940 veces más grande y en otro continente

Todo lo que hiciste aquí es lo que ocurre del otro lado cuando llamas a la API de Groq en el
notebook 09. Conviene tener claro qué cambia y qué no.

| | GPT-2 español (aquí) | `openai/gpt-oss-120b` (por la API) |
|---|---|---|
| Parámetros | 124 millones | 117 mil millones (activa ~5 mil millones por token: es un modelo de *mezcla de expertos*) |
| Ventana de contexto | 1,024 tokens | 131,072 tokens |
| Dónde corre | Tu computadora | El servidor del proveedor |
| ¿Ves los logits? | Sí, completos | No: solo el texto elegido |
| Entrenamiento | Solo predecir el siguiente token | Lo mismo + ajuste por instrucciones y preferencias humanas |

Lo que **no** cambia: tokens, tabla de embeddings, capas de atención causal, logits, *softmax*,
temperatura y el bucle de la sección 6. El modelo grande es el mismo mecanismo con más capas, más
datos y una fase extra de entrenamiento que lo vuelve obediente a instrucciones.

Lo que **sí** cambia en la práctica es que dejas de ver la distribución. Recibes un token ya
elegido, sin la información de si su probabilidad era 0.95 o 0.04. **El modelo se ve igual de
seguro en los dos casos.** Ésa es, mecánicamente, la razón de que las alucinaciones suenen tan
convincentes.

### Dónde encaja todo lo demás del módulo

Los tres roles del mensaje (`system`, `user`, `assistant`), el contexto que recupera tu sistema
RAG y el historial de la conversación **no son mecanismos distintos**: los tres terminan
concatenados en un solo texto que se tokeniza y se mete por el mismo tubo que acabas de recorrer.

In [ ]:
# El prompt de sistema real de la app de la sesión 11 (rag_core.py).
prompt_sistema = """Eres el asistente del Módulo V del Diplomado de Ciencia de Datos de la FES Acatlán, UNAM.

Reglas que debes respetar siempre:
1. Responde ÚNICAMENTE con la información de los fragmentos de CONTEXTO que recibes.
2. Cita entre corchetes el número del fragmento que respalda cada afirmación, así: [1].
3. Si el contexto no contiene la respuesta, dilo con claridad y no la completes con
   conocimiento propio. Es preferible admitir que no sabes a inventar.
4. Responde en español, en un tono claro y didáctico, en un máximo de seis oraciones.
5. No inventes números, fechas ni nombres que no estén en el contexto."""

pregunta = "¿Para qué sirve el coeficiente de silueta?"

print("prompt de sistema : %4d tokens  (en CADA llamada, aunque no cambie)" % len(tok(prompt_sistema)["input_ids"]))
print("pregunta del usuario: %2d tokens" % len(tok(pregunta)["input_ids"]))
print("\nSi recuperas 3 fragmentos de ~120 tokens cada uno, el modelo recibe")
print("unos", len(tok(prompt_sistema)["input_ids"]) + len(tok(pregunta)["input_ids"]) + 360,
      "tokens de entrada para una pregunta de 9.")

Un prompt de sistema de 161 tokens se paga **en cada llamada**, aunque nunca cambie. Con el
contexto recuperado, una pregunta de 9 tokens se vuelve una entrada de más de 500. Eso es lo que
hace RAG: **gastar ventana de contexto a cambio de que el modelo no tenga que adivinar.**

Y explica por qué no se resuelve el problema metiendo el libro completo en el prompt. No es que
esté prohibido: es que la ventana es finita, se paga por token, y —como sabes desde el notebook
06— darle texto irrelevante empeora la respuesta.

## 8. Tu turno

### Ejercicio 1 — ¿De qué está seguro el modelo?

Escribe dos *prompts*: uno cuyo final sea casi obligado y otro genuinamente abierto. Compara la
probabilidad del token más probable y cuántos tokens hacen falta para juntar el 90 %.

Con los dos ejemplos que vienen cargados deberías ver una diferencia enorme: **36 tokens contra
más de 4,000** para cubrir la misma masa de probabilidad. Ojo con los refranes y las frases
hechas: este modelo es chico y muchas no las conoce, así que sirven para otra discusión — cuando
el modelo *no* sabe, no se queda callado, simplemente reparte la probabilidad.

In [ ]:
def diagnostico(prompt):
    """Devuelve la confianza del modelo sobre el siguiente token."""
    enc = tok(prompt, return_tensors="pt")
    with torch.no_grad():
        lg = modelo(**enc).logits[0, -1]
    p = torch.softmax(lg, -1)
    orden = torch.sort(p, descending=True).values
    return {
        "prompt": prompt,
        "token más probable": repr(tok.decode([int(lg.argmax())])),
        "su probabilidad": round(float(p.max()), 3),
        "tokens para el 90%": int((torch.cumsum(orden, 0) < 0.9).sum()) + 1,
    }

mis_prompts = [
    "Universidad Nacional Autónoma de",   # final casi obligado
    "El principal problema de",           # genuinamente abierto
    # TODO: agrega dos prompts tuyos
]
pd.DataFrame([diagnostico(p) for p in mis_prompts])

### Ejercicio 2 — El costo de escribir en español

La sección 1.1 midió dos oraciones. Mide un párrafo tuyo —de tu tarea, de tu trabajo— con los dos
tokenizadores y calcula el sobrecosto. Si el proveedor cobrara \$1 por cada 1,000 tokens, ¿cuánto
te costaría de más ese párrafo por estar en español?

In [ ]:
mi_parrafo = """
Escribe aquí un párrafo tuyo, de unas cinco o seis líneas.
"""

n_es = len(tok(mi_parrafo)["input_ids"])
n_en = len(tok_en(mi_parrafo)["input_ids"])
print(f"palabras                      : {len(mi_parrafo.split())}")
print(f"tokens (tokenizador español)  : {n_es}")
print(f"tokens (tokenizador inglés)   : {n_en}")
print(f"sobrecosto                    : {100*(n_en-n_es)/n_es:+.0f}%")

# TODO: ¿cuántos tokens tiene el prompt de sistema de TU aplicación?
#       ¿Se puede decir lo mismo en menos tokens sin perder las reglas?

## Para pensar

1. En la sección 4 el modelo repartió el 58 % de la probabilidad fuera de sus 10 mejores
   candidatos. Cuando la API te devuelve el texto ya elegido, esa información desaparece.
   ¿Qué implica para un sistema que responde preguntas de personas que no pueden verificar la
   respuesta? ¿Cómo se relaciona con la regla 3 del prompt de sistema de la app?

2. En 3.1 los vectores de `banco` eran idénticos al entrar y distintos al salir. Si en vez de
   `banco` la palabra fuera `positivo` en un análisis de sentimiento, ¿qué ventaja te daría un
   embedding contextual frente a la bolsa de palabras de la Sesión 8?

3. La temperatura no agrega información: solo deforma la distribución que el modelo ya calculó.
   Entonces, ¿por qué `temperature=0` tampoco garantiza que la respuesta sea correcta?

4. El bucle de la sección 6 escribe sin plan, de izquierda a derecha. ¿Cómo explica eso que un
   modelo sostenga un error en vez de corregirlo a media respuesta? ¿Y qué le pedirías al prompt
   de sistema para reducirlo?

5. Un documento en español consume ~67 % más tokens que su traducción al inglés con un tokenizador
   entrenado en inglés. Si tu corpus de RAG está en español y la ventana de contexto es fija,
   ¿qué decisiones de diseño cambia eso? (Piensa en el tamaño de los fragmentos y en el $k$ de la
   recuperación.)

6. En 3.2 comprobaste que agregar texto al final no cambia lo ya calculado. Sabiendo eso, ¿cómo
   ordenarías las partes de un prompt (instrucciones fijas, documentos recuperados, pregunta del
   usuario) para aprovechar el caché del proveedor?

---

**Sigue en `08 Building-a-Chat/`**, donde este mismo mecanismo se usa por API con un modelo grande
y se le agrega memoria y herramientas; y en `09 RAG-Completo/`, donde se cierra el ciclo de RAG
con generación.